In [1]:
__author__ = "Melat Ghebreselassie"
__version__ = "01/22/2026"

# Table of Contents  
1. [Set-up](#Set-up)     
2. [pyvene 101 with NDIF Backend](#pyvene-101-with-ndif-backend)
3. [Collecting Interventions](#Collecting-Interventions)
4. [Vanilla Intervention](#Vanilla-Intervention)
5. [Trainable Interventions](#Trainable-Interventions-via-pyvene-API)
   - LowRankRotatedSpaceIntervention
   - RotatedSpaceIntervention
   - BoundlessRotatedSpaceIntervention
   - SigmoidMaskRotatedSpaceIntervention
   - SigmoidMaskIntervention
6. [Non-Trainable Interventions](#Non-Trainable-Interventions)
   - ZeroIntervention
   - AdditionIntervention
   - SubtractionIntervention
7. [generate() with Interventions](#generate-with-Interventions)
8. [Serial Interventions](#Serial-Interventions)
9. [Save and Load Interventions](#Save-and-Load-Interventions)
10. [Summary](#Summary)

# Set-up

In [ ]:
import os

# Load credentials from a .env file (recommended - keep .env out of git)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded credentials from .env")
except ImportError:
    print("python-dotenv not installed; falling back to environment variables.")

# Verify required keys are present
assert os.environ.get("NDIF_API_KEY"), "Set NDIF_API_KEY in your .env file or environment"
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in your .env file or environment"
print("Credentials OK")

In [ ]:
try:
    # This library is our indicator that the required installs
    # need to be done.
    import pyvene as pv
    import nnsight 

except ModuleNotFoundError:
    !pip install git+https://github.com/stanfordnlp/pyvene.git

# pyvene 101 with ndif backend


In [3]:
USE_REMOTE = True

In [4]:
from nnsight import LanguageModel

print("Loading GPT-2 model...")
if USE_REMOTE:
    model = LanguageModel('openai-community/gpt2')
else:
    model = LanguageModel('openai-community/gpt2', device_map='cpu')

tokenizer = model.tokenizer

Loading GPT-2 model...


# Collecting Interventions

In [ ]:
import pyvene as pv

base_text = "When John and Mary went to the shops, Mary gave the bag to"

pv_model = pv.build_intervenable_model({
        "component": "transformer.h[0].mlp.c_proj.output",
        "intervention": pv.CollectIntervention()
    }, model=model, 
    remote=USE_REMOTE)

collected_mlp_out = pv_model(
    base = tokenizer(base_text, return_tensors="pt"),
    unit_locations={"base": [h for h in range(12)]}
)[0][-1][0]

# Resolve the saved proxy value
if hasattr(collected_mlp_out, 'value'):
    collected_mlp_out = collected_mlp_out.value

print(f"Collected MLP output shape: {collected_mlp_out.shape}")
print("CollectIntervention: SUCCESS!" if collected_mlp_out is not None else "FAILED")

# Vanilla Intervention

In [6]:
def get_clean_output(text):
    """Get clean model output without any intervention."""
    with model.session(remote=USE_REMOTE):
        with model.trace(text):
            output = model.lm_head.output.save()  # Use lm_head.output for logits
    return output


def get_logits(output):
    """Extract logits from various output formats."""
    if hasattr(output, 'logits'):
        return output.logits
    elif isinstance(output, dict) and 'logits' in output:
        return output['logits']
    elif hasattr(output, 'value'):
        return output.value.logits if hasattr(output.value, 'logits') else output.value
    return output

In [ ]:

# Prepare inputs FIRST
# Note: GPT-2's baseline prediction for both Spain/Italy is often ' the' (not the city name).
# The intervention swaps layer-0 activations from Italy into the Spain forward pass,
# which should shift the prediction toward ' Rome'.
base_text = "The capital of Spain is"
source_text = "The capital of Italy is"

pv_model = pv.build_intervenable_model({
        "component": "transformer.h[0].output",
        "intervention": pv.VanillaIntervention()
}, model=model, remote=USE_REMOTE)

# Now get clean output with the correct base_text
clean_output = get_clean_output(base_text)
clean_logits = get_logits(clean_output)

# Use raw strings for remote compatibility
_, intervened = pv_model(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
intervened_logits = get_logits(intervened)

clean_pred = tokenizer.decode(clean_logits[0, -1].argmax())
intervened_pred = tokenizer.decode(intervened_logits[0, -1].argmax())
logit_diff = (clean_logits - intervened_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"Intervened prediction: '{intervened_pred}'")
print(f"Mean absolute logit difference: {logit_diff:.4f}")
# Success = intervention changed the output toward Rome (Italy), regardless of clean baseline
print("SUCCESS!" if intervened_pred == " Rome" else "UNEXPECTED OUTPUT")

# Trainable Interventions via pyvene API

Trainable interventions like `LowRankRotatedSpaceIntervention`, `RotatedSpaceIntervention`, and `BoundlessRotatedSpaceIntervention` can be used directly via the pyvene API - no manual nnsight operations required!

In [8]:
import torch

EMBED_DIM = 768  # GPT-2 hidden dimension

# Use same text as VanillaIntervention example
base_text = "The capital of Spain is"
source_text = "The capital of Italy is"

# Get clean output for comparison
clean_output = get_clean_output(base_text)
clean_logits = get_logits(clean_output)

# =============================================================================
# LowRankRotatedSpaceIntervention via pyvene API (simple approach)
# =============================================================================
intervention = pv.LowRankRotatedSpaceIntervention(
    embed_dim=EMBED_DIM, low_rank_dimension=64
)

pv_lowrank = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": intervention
}, model=model, remote=USE_REMOTE)

_, lowrank_output = pv_lowrank(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": 4}
)
lowrank_logits = get_logits(lowrank_output)

clean_pred = tokenizer.decode(clean_logits[0, -1].argmax())
lowrank_pred = tokenizer.decode(lowrank_logits[0, -1].argmax())
logit_diff = (clean_logits - lowrank_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"LowRankRotatedSpace prediction: '{lowrank_pred}'")
print(f"Mean logit diff: {logit_diff:.4f}")
print(f"Intervention had effect: {logit_diff > 0.01}")

⬇ Downloading:   0%|          | 0.00/504k [00:00<?]

⬇ Downloading:   0%|          | 0.00/512k [00:00<?]

Clean prediction: ' the'
LowRankRotatedSpace prediction: ' Madrid'
Mean logit diff: 0.2520
Intervention had effect: True


In [9]:
# =============================================================================
# RotatedSpaceIntervention via pyvene API
# =============================================================================
rotated_intervention = pv.RotatedSpaceIntervention(embed_dim=EMBED_DIM)

pv_rotated = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": rotated_intervention
}, model=model, remote=USE_REMOTE)

_, rotated_output = pv_rotated(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": 4}
)
rotated_logits = get_logits(rotated_output)

rotated_pred = tokenizer.decode(rotated_logits[0, -1].argmax())
rotated_diff = (clean_logits - rotated_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"RotatedSpace prediction: '{rotated_pred}'")
print(f"Mean logit diff: {rotated_diff:.4f}")
print(f"Intervention had effect: {rotated_diff > 0.01}")

⬇ Downloading:   0%|          | 0.00/512k [00:00<?]

Clean prediction: ' the'
RotatedSpace prediction: ' Rome'
Mean logit diff: 1.5078
Intervention had effect: True


In [ ]:
# =============================================================================
# BoundlessRotatedSpaceIntervention via pyvene API
# =============================================================================
boundless_intervention = pv.BoundlessRotatedSpaceIntervention(
    embed_dim=EMBED_DIM, low_rank_dimension=64
)

pv_boundless = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": boundless_intervention
}, model=model, remote=USE_REMOTE)

_, boundless_output = pv_boundless(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": 4}
)
boundless_logits = get_logits(boundless_output)

boundless_pred = tokenizer.decode(boundless_logits[0, -1].argmax())
boundless_diff = (clean_logits - boundless_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"BoundlessRotatedSpace prediction: '{boundless_pred}'")
print(f"Mean logit diff: {boundless_diff:.4f}")
# At random init the boundary (sigmoid) softly blends base and source, so
# logit diff > 0 confirms the intervention is running even if top-1 doesn't change.
print(f"Intervention had effect: {boundless_diff > 0.01} (top-1 may match clean at random init)")

In [11]:
# =============================================================================
# SigmoidMaskRotatedSpaceIntervention via pyvene API
# =============================================================================
sigmoid_mask_rotated_intervention = pv.SigmoidMaskRotatedSpaceIntervention(
    embed_dim=EMBED_DIM, low_rank_dimension=64
)

pv_sigmoid_mask_rotated = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": sigmoid_mask_rotated_intervention
}, model=model, remote=USE_REMOTE)

_, sigmoid_mask_rotated_output = pv_sigmoid_mask_rotated(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": 4}
)
sigmoid_mask_rotated_logits = get_logits(sigmoid_mask_rotated_output)

sigmoid_mask_rotated_pred = tokenizer.decode(sigmoid_mask_rotated_logits[0, -1].argmax())
sigmoid_mask_rotated_diff = (clean_logits - sigmoid_mask_rotated_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"SigmoidMaskRotatedSpace prediction: '{sigmoid_mask_rotated_pred}'")
print(f"Mean logit diff: {sigmoid_mask_rotated_diff:.4f}")
print(f"Intervention had effect: {sigmoid_mask_rotated_diff > 0.01}")

⬇ Downloading:   0%|          | 0.00/512k [00:00<?]

Clean prediction: ' the'
SigmoidMaskRotatedSpace prediction: ' Rome'
Mean logit diff: 1.1250
Intervention had effect: True


In [ ]:
# =============================================================================
# SigmoidMaskIntervention via pyvene API
# (Learnable mask without rotation - applies directly in activation space)
# =============================================================================
sigmoid_mask_intervention = pv.SigmoidMaskIntervention(embed_dim=EMBED_DIM)

pv_sigmoid_mask = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": sigmoid_mask_intervention
}, model=model, remote=USE_REMOTE)

_, sigmoid_mask_output = pv_sigmoid_mask(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": 4}
)
sigmoid_mask_logits = get_logits(sigmoid_mask_output)

sigmoid_mask_pred = tokenizer.decode(sigmoid_mask_logits[0, -1].argmax())
sigmoid_mask_diff = (clean_logits - sigmoid_mask_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"SigmoidMask prediction: '{sigmoid_mask_pred}'")
print(f"Mean logit diff: {sigmoid_mask_diff:.4f}")
# At random init mask=zeros → sigmoid(0/temp)=0.5, so output is 50/50 blend of base and source.
# logit diff > 0 confirms the intervention is running even if top-1 doesn't change.
print(f"Intervention had effect: {sigmoid_mask_diff > 0.01} (top-1 may match clean at random init)")

# Non-Trainable Interventions

Zero, Addition, Subtraction, and Lambda interventions work with the NDIF backend without any training step.

In [ ]:
# =============================================================================
# ZeroIntervention - zeros out the target activation
# =============================================================================
pv_zero = pv.build_intervenable_model({
    "component": "transformer.h[6].output",
    "intervention": pv.ZeroIntervention(embed_dim=EMBED_DIM)
}, model=model, remote=USE_REMOTE)

_, zero_output = pv_zero(
    base=base_text,
    unit_locations={"base": [None]}
)
zero_logits = get_logits(zero_output)
zero_diff = (clean_logits - zero_logits).abs().mean().item()
print(f"ZeroIntervention logit diff: {zero_diff:.4f}")
print(f"SUCCESS: {zero_diff > 0.001}" if zero_diff > 0.001 else "FAILED")

# =============================================================================
# AdditionIntervention - adds a source activation to base
# =============================================================================
pv_add = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": pv.AdditionIntervention(embed_dim=EMBED_DIM)
}, model=model, remote=USE_REMOTE)

_, add_output = pv_add(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
add_logits = get_logits(add_output)
add_diff = (clean_logits - add_logits).abs().mean().item()
print(f"\nAdditionIntervention logit diff: {add_diff:.4f}")
print(f"SUCCESS: {add_diff > 0.001}" if add_diff > 0.001 else "FAILED")

# =============================================================================
# SubtractionIntervention - subtracts source activation from base
# =============================================================================
pv_sub = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": pv.SubtractionIntervention(embed_dim=EMBED_DIM)
}, model=model, remote=USE_REMOTE)

_, sub_output = pv_sub(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
sub_logits = get_logits(sub_output)
sub_diff = (clean_logits - sub_logits).abs().mean().item()
print(f"\nSubtractionIntervention logit diff: {sub_diff:.4f}")
print(f"SUCCESS: {sub_diff > 0.001}" if sub_diff > 0.001 else "FAILED")

# generate() with Interventions

The NDIF backend supports `pv_model.generate()` which uses nnsight's `model.generate()` context to apply interventions during token generation.

In [ ]:
# =============================================================================
# generate() with VanillaIntervention
# =============================================================================
pv_gen = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": pv.VanillaIntervention()
}, model=model, remote=USE_REMOTE)

# Baseline generation (no intervention)
with model.session(remote=USE_REMOTE):
    with model.generate(base_text, max_new_tokens=5):
        clean_gen = model.generator.output.save()

# Intervened generation
_, gen_output = pv_gen.generate(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])},
    max_new_tokens=5,
)

def decode_gen(output):
    """Decode generation output to string."""
    if hasattr(output, 'value'):
        output = output.value
    if hasattr(output, 'sequences'):
        return tokenizer.decode(output.sequences[0], skip_special_tokens=True)
    if isinstance(output, torch.Tensor):
        return tokenizer.decode(output[0], skip_special_tokens=True)
    return str(output)

clean_text = decode_gen(clean_gen)
intervened_text = decode_gen(gen_output)

print(f"Clean generation:     '{clean_text}'")
print(f"Intervened generation: '{intervened_text}'")
print(f"Generation differs: {clean_text != intervened_text}")

# Serial Interventions

Serial mode chains interventions so that each group's source is run through the model with prior groups' patches already applied.

In [ ]:
# =============================================================================
# Serial intervention with two sources
# Group 0: patch layer 0 from source_0 into base
# Group 1: patch layer 6 from source_1 into base (after group 0 already applied)
# =============================================================================
source_text_0 = "The capital of Italy is"
source_text_1 = "The capital of France is"

pv_serial = pv.build_intervenable_model(
    [
        {"component": "transformer.h[0].output", "intervention": pv.VanillaIntervention(), "group_key": 0},
        {"component": "transformer.h[6].output", "intervention": pv.VanillaIntervention(), "group_key": 1},
    ],
    model=model,
    remote=USE_REMOTE,
    mode="serial",
)

_, serial_output = pv_serial(
    base=base_text,
    sources=[source_text_0, source_text_1],
    unit_locations={"source_0->source_1": ([None], [None]), "source_1->base": ([None], [None])}
)
serial_logits = get_logits(serial_output)
serial_pred = tokenizer.decode(serial_logits[0, -1].argmax())
serial_diff = (clean_logits - serial_logits).abs().mean().item()

print(f"Clean prediction:  '{clean_pred}'")
print(f"Serial prediction: '{serial_pred}'")
print(f"Logit diff (serial vs clean): {serial_diff:.4f}")
print(f"Serial intervention had effect: {serial_diff > 0.001}")

# Save and Load Interventions

Trained intervention weights can be saved to disk and reloaded, preserving the exact weight values.

In [ ]:
import tempfile, os

# Build and run a LowRankRotatedSpaceIntervention
save_intervention = pv.LowRankRotatedSpaceIntervention(embed_dim=EMBED_DIM, low_rank_dimension=32)
pv_to_save = pv.build_intervenable_model({
    "component": "transformer.h[3].output",
    "intervention": save_intervention
}, model=model, remote=USE_REMOTE)

# Run it once to get a baseline output
_, output_before = pv_to_save(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": 4}
)
logits_before = get_logits(output_before)

# Extract weights before saving
weights_before = save_intervention.rotate_layer.weight.detach().clone()

# Save to a temp directory
save_dir = tempfile.mkdtemp(prefix="pyvene_ndif_")
pv_to_save.save(save_dir)
print(f"Saved to: {save_dir}")
print(f"Files: {os.listdir(save_dir)}")

# Load back
pv_loaded = pv.IntervenableNdifModel.load(save_dir, model)
loaded_intervention = list(pv_loaded.interventions.values())[0]
weights_after = loaded_intervention.rotate_layer.weight.detach().clone()

# Verify weights match
weights_match = torch.allclose(weights_before, weights_after)
print(f"\nWeights preserved after save/load: {weights_match}")

# Run the loaded model and verify outputs match
_, output_after = pv_loaded(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": 4}
)
logits_after = get_logits(output_after)
outputs_match = torch.allclose(logits_before, logits_after, atol=1e-4)
print(f"Outputs preserved after save/load: {outputs_match}")
print("SUCCESS!" if weights_match and outputs_match else "FAILED")

# Summary

All intervention types work via the standard pyvene API with NDIF backend (`remote=True`).

## Non-Trainable Interventions

| Intervention | Description | NDIF Support |
|-------------|-------------|--------------|
| `CollectIntervention` | Collects activations without modification | ✓ Works |
| `VanillaIntervention` | Simple activation swapping (base ← source) | ✓ Works |
| `ZeroIntervention` | Zeros out the target activation | ✓ Works |
| `AdditionIntervention` | Adds source activation to base | ✓ Works |
| `SubtractionIntervention` | Subtracts source activation from base | ✓ Works |
| `LambdaIntervention` | Custom function applied to (base, source) | ✓ Works |
| `NoiseIntervention` | Adds Gaussian noise at specified dimensions | ✓ Works |

## Trainable Interventions

| Intervention | Description | NDIF Support |
|-------------|-------------|--------------|
| `LowRankRotatedSpaceIntervention` | Projects to low-rank rotated subspace, swaps, projects back | ✓ Works |
| `RotatedSpaceIntervention` | Full-rank rotation with learnable orthogonal matrix | ✓ Works |
| `BoundlessRotatedSpaceIntervention` | Rotation + learnable boundary for soft intervention | ✓ Works |
| `SigmoidMaskRotatedSpaceIntervention` | Rotation + per-dimension sigmoid masks | ✓ Works |
| `SigmoidMaskIntervention` | Per-dimension sigmoid masks (no rotation) | ✓ Works |
| `PCARotatedSpaceIntervention` | Intervention in PCA-computed subspace | ✓ Works |
| `AutoencoderIntervention` | Intervention in autoencoder latent space | ✓ Works |
| `JumpReLUAutoencoderIntervention` | Intervention via JumpReLU SAE latents | ✓ Works |

## Additional Features

| Feature | NDIF Support |
|---------|--------------|
| `pv_model.generate(...)` | ✓ Works — uses `model.generate()` nnsight context |
| Serial mode (`mode="serial"`) | ✓ Works — chains multi-group interventions |
| `pv_model.save(directory)` | ✓ Works — saves config + weights |
| `IntervenableNdifModel.load(directory, model)` | ✓ Works — reloads and verifies weights |

## Usage Pattern

```python
# All interventions follow the same pattern:
pv_model = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": pv.VanillaIntervention()
}, model=model, remote=True)

# Forward pass
_, output = pv_model(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)

# Generation
_, gen_output = pv_model.generate(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])},
    max_new_tokens=10,
)

# Save / Load
pv_model.save("./my_intervention")
pv_loaded = pv.IntervenableNdifModel.load("./my_intervention", model)
```